# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Our model provides a directional decision-support score (probability of decay). Based on this score, we rank the backlog.
The main action is always review_for_refresh. To build trust, we assign simple reason codes:

high_probability_decay (ML score > 0.7): Urgent review, strong measured signals of decline.

moderate_decay_risk (ML score 0.5 - 0.7): Secondary priority.

standard_review (ML score < 0.5): Healthy or inconclusive.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Quick data prep and model train (using our Week 5 approach)
df['impressions_90d'] = df['impressions_90d'].fillna(0)
df['sessions_90d'] = df['sessions_90d'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(df['content_age_days'].median())
df['is_declining'] = df['trend_direction'] == 'down'

features = ['impressions_90d', 'sessions_90d', 'content_age_days']
model = RandomForestClassifier(max_depth=5, random_state=42)
model.fit(df[features], df['is_declining'])

# Generate scores
df['decay_probability'] = model.predict_proba(df[features])[:, 1]

# Apply actions and reason codes
df['action'] = 'review_for_refresh'
df['reason_code'] = 'standard_review'
df.loc[df['decay_probability'] > 0.7, 'reason_code'] = 'high_probability_decay'
df.loc[(df['decay_probability'] <= 0.7) & (df['decay_probability'] > 0.5), 'reason_code'] = 'moderate_decay_risk'

# Create the ranked queue
queue = df.sort_values('decay_probability', ascending=False).copy()
print("Top 10 Ranked Actions:")
display(queue[['content_id', 'decay_probability', 'reason_code', 'action']].head(10))

Top 10 Ranked Actions:


,content_id,decay_probability,reason_code,action
21722,content_2cadea20ede8,0.746009,high_probability_decay,review_for_refresh
6797,content_1e57b71527e4,0.744818,high_probability_decay,review_for_refresh
25108,content_5a467b5c7648,0.744362,high_probability_decay,review_for_refresh
12774,content_08115d3a65f6,0.742295,high_probability_decay,review_for_refresh
3479,content_4c8b6a410f88,0.740375,high_probability_decay,review_for_refresh
20944,content_0f1592ba330d,0.740375,high_probability_decay,review_for_refresh
3572,content_018ebe1e96c0,0.740375,high_probability_decay,review_for_refresh
10606,content_88a2e9238054,0.740375,high_probability_decay,review_for_refresh
6883,content_b012f02fa8f1,0.740375,high_probability_decay,review_for_refresh
13455,content_257f801f1cd1,0.738998,high_probability_decay,review_for_refresh


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use: This playbook is for Content Strategists and SEO Managers. It is meant to prioritize a massive backlog of thousands of pages into a manageable daily review queue.
Limits: It stops being valid during massive external shifts. The model only looks at observed historical traffic and age. It does not know if a competitor just launched a better page, or if a Google Core Update completely changed the SERP layout.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended Users: Content Strategists, SEO Editors.")
print("Core Limit: The model lacks external context (seasonality, competitor actions, algorithm updates).")
print("It is a prioritization engine, not a replacement for human SEO expertise.")

Intended Users: Content Strategists, SEO Editors.
Core Limit: The model lacks external context (seasonality, competitor actions, algorithm updates).
It is a prioritization engine, not a replacement for human SEO expertise.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review: A person must manually check the live search results to see why traffic dropped. Did search intent change? Is the content factually outdated?
The No-Go List: We should never automate the actual rewriting, merging, or unpublishing of content based solely on this score. The score is a flag, not a final verdict.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Human Checklist before acting:")
print("1. Verify current SERP intent.")
print("2. Check for seasonal trends (e.g., holiday content).")
print("\nNO-GO AUTOMATION:")
print("- Never auto-delete or auto-redirect pages based on this score.")

Human Checklist before acting:
1. Verify current SERP intent.
2. Check for seasonal trends (e.g., holiday content).

NO-GO AUTOMATION:
- Never auto-delete or auto-redirect pages based on this score.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

We know the recommendations have gone stale if the precision of our top 100 queue drops significantly (e.g., if humans review the top 100 pages and find most of them are actually healthy).
Retrain triggers:

Time-based: Retrain every 6 months with fresh data.

Event-based: Retrain 30 days after a major confirmed Google Core Update, once metrics settle.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Monitoring Metric: Human acceptance rate of the top 100 queue.")
print("Retrain Triggers:")
print("- Every 6 months.")
print("- Post-Google Core Update (after a 30-day settling period).")

Monitoring Metric: Human acceptance rate of the top 100 queue.
Retrain Triggers:
- Every 6 months.
- Post-Google Core Update (after a 30-day settling period).


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

I am exporting the final ranked queue to our outputs folder. This CSV file is the tangible deliverable that a content team would ingest into their workflow, and it serves as the foundation for our final Capstone write-up.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Successfully exported {len(queue)} rows to work/outputs/action_playbook_queue.csv")

Successfully exported 30000 rows to work/outputs/action_playbook_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.